# Loading Data And QC

**REQUIRED DAY 2**

## From count matrix to AnnData

Everything from here on works with **AnnData** — the data structure `scanpy` is built on: a cell-by-gene matrix (`.X`) plus per-cell metadata (`.obs`) and per-gene metadata (`.var`) that travel together as one object.

Read today's shared checkpoint in:

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/tscc/nfs/home/juf009/day2_shared_data/counts/checkpoint.h5ad")
adata


## Explore the object before you do anything else

Before touching QC, get oriented the same way you would with any new dataset: look at what's actually inside it. Fill in each cell below (one line each) rather than skipping ahead — this is the exact habit the BMI710 course opens with too, and for the same reason: an object you haven't inspected is an object you're guessing about.

- Print the first 5 cell barcodes (`.obs_names`).
- Print the first 5 gene symbols (`.var_names`).
- Check the type of `adata.X` (hint: `type(...)`) — is it a dense array or a sparse matrix? Why would sparse matter here (see Lucas-style reasoning: how many of these values are actually zero)?
- Print `adata.shape` — how many cells, how many genes?

In [ ]:
# adata.obs_names[:5]



In [ ]:
# adata.var_names[:5]



In [ ]:
# type(adata.X)



In [ ]:
# adata.shape



Confirm `adata.n_obs` and `adata.n_vars` are in the range you expect *before* doing anything else — this is exactly what [templates/diagnostic_scripts/verify_counts_matrix.py](../templates/diagnostic_scripts/verify_counts_matrix.py) automates.

## Quality control metrics

Per-cell QC in scanpy centers on three numbers:

- **`n_genes_by_counts`** — how many distinct genes were detected in this cell. Very low: likely an empty droplet or dying cell. Very high: possibly a doublet (see below).
- **`total_counts`** — total UMIs per cell. Same logic as above.
- **`pct_counts_mt`** — percent of counts from mitochondrial genes. High mitochondrial fraction is a classic signature of a dying or ruptured cell (cytoplasmic RNA leaks out, mitochondrial RNA is relatively retained).

Flag mitochondrial genes yourself first — human mitochondrial gene symbols all start with `MT-`. Write the boolean column, then look up scanpy's function for computing QC metrics from a set of flagged genes ([`sc.pp.calculate_qc_metrics`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.calculate_qc_metrics.html) — check the docs for the `qc_vars` argument) and run it.

In [ ]:
## Fill in: flag mitochondrial genes into adata.var["mt"], then call sc.pp.calculate_qc_metrics
## with qc_vars=["mt"], percent_top=None, log1p=False, inplace=True


adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()


Now plot the three QC metrics. Look up the scanpy plotting function for a violin plot of one or more `.obs` columns ([`sc.pl.violin`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.violin.html)).

In [ ]:
## Fill in: sc.pl.violin(...) for ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4



## Doublet flagging is a QC step, not an afterthought

A "doublet" is two cells captured in one droplet and sequenced as if they were one cell — it will look like a real cell with QC metrics that pass every threshold above, but its transcriptome is a mixture of two cell types. Flag it explicitly rather than hoping clustering will sort it out later. Look up scanpy's built-in doublet detector ([`sc.pp.scrublet`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.scrublet.html)) and run it, then check how many cells it flagged.

In [ ]:
## Fill in: sc.pp.scrublet(adata), then look at adata.obs["predicted_doublet"].value_counts()



Doublets don't announce themselves in the QC metrics above — a doublet can pass every threshold and still be two cells' worth of transcriptome mixed together. Checking for it explicitly, rather than assuming clustering will sort it out later, is the point of this step.

## Practice

Look at the histograms above yourself and write your own filtering thresholds for `n_genes_by_counts`, `total_counts`, and `pct_counts_mt`, with one sentence of reasoning for each based on what you actually see in *this* dataset's distributions — not a number copied from a tutorial. Apply them in the cell below.

In [ ]:
# Apply your QC filtering thresholds here, once you've written down your reasoning above.




## Save your checkpoint

Each notebook in this sequence opens with its own fresh kernel, so `adata` doesn't carry over by itself -- this saves it to disk, and 06 loads it back in. Same reasoning as every checkpoint file elsewhere in this bootcamp.

In [ ]:
import os

os.makedirs("results", exist_ok=True)
adata.write_h5ad("results/checkpoint_05_qc.h5ad")
print("Saved to results/checkpoint_05_qc.h5ad")


## Further reading

- [Single-cell best practices — Quality Control](https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html)
- [scanpy: Preprocessing and clustering tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html)
- [Seurat's essential commands](https://satijalab.org/seurat/articles/essential_commands.html) — if you're curious what the equivalent R/Seurat workflow looks like for everything in this notebook.